# API Fundamentals: GET, Params, and Error Handling

In [1]:
import json
from urllib.parse import urlparse, parse_qs

import requests

```{contents}
:local:
:depth: 2
```

Web APIs let Python programs request data from services over HTTP. In data work, an API is often the difference between copying a file by hand and writing a repeatable data collection step. This section focuses on read-only API calls: choosing an endpoint, passing query parameters, reading JSON, and handling errors without crashing a larger analysis.

## What Is an API?

An **application programming interface** (API) is a defined way for one program to ask another program for a service. A **web API** exposes that interface through URLs. A **REST-style API** usually treats each URL as a resource and uses HTTP methods to say what kind of action the client wants.

| Method | Common purpose | Example use |
|---|---|---|
| `GET` | Retrieve data | Get current weather for a city |
| `POST` | Send new data | Submit a form or create a record |
| `PUT` or `PATCH` | Update data | Change a profile setting |
| `DELETE` | Remove data | Delete a saved item |

Status codes summarize how the request went:

| Code range | Meaning | Typical action |
|---|---|---|
| `2xx` | Success | Parse the response |
| `3xx` | Redirect | Usually handled by the client library |
| `4xx` | Client-side problem | Check the endpoint, parameters, or authentication |
| `5xx` | Server-side problem | Retry later or report failure gracefully |

A useful habit is to describe every API call as **endpoint + method + parameters + expected response shape**.

### Worked Example: Read an API Contract

Suppose a weather endpoint documents this contract:

| Field | Value |
|---|---|
| Endpoint | `https://api.example.test/weather` |
| Method | `GET` |
| Required params | `city`, `units` |
| Response | JSON object with `city`, `temperature`, and `units` |

Before writing code, identify the smallest successful request and the keys your code expects back.

In [2]:
api_contract = {
    "endpoint": "https://api.example.test/weather",
    "method": "GET",
    "required_params": ["city", "units"],
    "response_keys": ["city", "temperature", "units"],
}

print(api_contract["method"], api_contract["endpoint"])
print("Required params:", ", ".join(api_contract["required_params"]))
print("Expected keys:", ", ".join(api_contract["response_keys"]))

GET https://api.example.test/weather
Required params: city, units
Expected keys: city, temperature, units


## Making GET Requests with `requests`

The `requests` library is the common Python tool for HTTP. A basic `GET` request looks like this:

```python
response = requests.get(url, timeout=10)
response.raise_for_status()
data = response.json()
```

This notebook avoids live network calls so the book builds reliably offline. The next example uses a tiny response object with the same methods your code uses from a real `requests.Response`: `raise_for_status()` and `json()`.

### Worked Example: Parse a JSON Response

In [3]:
class DemoResponse:
    def __init__(self, payload, status_code=200):
        self._payload = payload
        self.status_code = status_code

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"HTTP {self.status_code}")

    def json(self):
        return self._payload

response = DemoResponse({"city": "Chicago", "temperature": 72.5, "units": "fahrenheit"})
response.raise_for_status()
data = response.json()

print(f"{data['city']}: {data['temperature']} degrees {data['units']}")

Chicago: 72.5 degrees fahrenheit


## Query Parameters

Many `GET` endpoints use **query parameters**: key/value pairs after the `?` in a URL. Build them with the `params=` argument instead of manually concatenating strings. `requests` handles escaping spaces, punctuation, and special characters correctly.

### Worked Example: Build a Request URL Safely

In [4]:
base_url = "https://api.example.test/weather"
params = {"city": "New York", "units": "fahrenheit"}

request = requests.Request("GET", base_url, params=params).prepare()
print(request.url)

parsed = urlparse(request.url)
print(parse_qs(parsed.query))

https://api.example.test/weather?city=New+York&units=fahrenheit
{'city': ['New York'], 'units': ['fahrenheit']}


### Checkpoint: Query Parameters

The endpoint `https://api.example.test/search` accepts `q`, `page`, and `limit`. Create a prepared `GET` request for page 2 of a search for `python api`, limited to 5 results. Print the final URL.

In [5]:
### Your code starts here
search_url = "https://api.example.test/search"
params = {
    # Fill in q, page, and limit.
}

prepared = requests.Request("GET", search_url, params=params).prepare()
print(prepared.url)
### Your code ends here

https://api.example.test/search


In [6]:
### Solution
search_url = "https://api.example.test/search"
params = {"q": "python api", "page": 2, "limit": 5}

prepared = requests.Request("GET", search_url, params=params).prepare()
print(prepared.url)

https://api.example.test/search?q=python+api&page=2&limit=5


## Error Handling

Two categories of errors matter in most API code:

1. **Network errors**: timeouts, DNS failures, broken connections, or other exceptions raised before a response arrives.
2. **HTTP errors**: the server responds, but the status code signals a problem, such as `404 Not Found` or `500 Internal Server Error`.

A reliable API helper should use a timeout, call `raise_for_status()`, and return a predictable value when the request fails.

### Worked Example: Safe JSON Fetching

In [7]:
def parse_json_response(response):
    """Return JSON data from a response-like object, or None if it fails."""
    try:
        response.raise_for_status()
        return response.json()
    except requests.HTTPError as error:
        print("HTTP error:", error)
    except ValueError as error:
        print("Invalid JSON:", error)
    return None

ok = DemoResponse({"results": ["a", "b"]}, status_code=200)
missing = DemoResponse({"message": "not found"}, status_code=404)

print(parse_json_response(ok))
print(parse_json_response(missing))

{'results': ['a', 'b']}
HTTP error: HTTP 404
None


## Weather API Pattern

A real weather call to Open-Meteo would use this shape:

```python
requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 41.88, "longitude": -87.63, "current_weather": True},
    timeout=10,
)
```

For a reproducible notebook, practice with a saved payload that has the same nested structure.

### Worked Example: Extract Weather Data

In [8]:
sample_weather = {
    "latitude": 41.88,
    "longitude": -87.63,
    "current_weather": {
        "temperature": 21.4,
        "windspeed": 13.2,
        "weathercode": 2,
    },
}

current = sample_weather["current_weather"]
print("Temperature:", current["temperature"])
print("Wind speed:", current["windspeed"])

Temperature: 21.4
Wind speed: 13.2


### Checkpoint: Weather Payload

Write a small function `summarize_weather(payload)` that returns a dictionary with the temperature and wind speed from a weather response payload.

In [9]:
def summarize_weather(payload):
    ### Your code starts here
    return {}
    ### Your code ends here

print(summarize_weather(sample_weather))

{}


In [10]:
### Solution
def summarize_weather(payload):
    current = payload["current_weather"]
    return {
        "temperature": current["temperature"],
        "windspeed": current["windspeed"],
    }

print(summarize_weather(sample_weather))

{'temperature': 21.4, 'windspeed': 13.2}


## Summary

API fundamentals are mostly about making contracts explicit. A `GET` request asks an endpoint for data, `params=` builds query strings safely, `response.json()` converts JSON into Python dictionaries and lists, and `raise_for_status()` turns failing HTTP status codes into exceptions your program can handle. For reliable notebooks and tests, separate the request step from the parsing step so parsing logic can be tested with saved payloads.